In [23]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub gdown')
    print("Setup complete!")


In [24]:
input_dir = 'datasets/preprocessed'

In [25]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import csv
import glob

print(f"Checking datasets in {input_dir}...")
if not os.path.exists(input_dir):
    print(f"Error: {input_dir} not found.")
else:
    required_keys = {"text", "label"}

    dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
    for input_file in dataset_files:
        print(f"\n--- Checking {os.path.basename(input_file)} ---")
        labels_found = set()
        error_records = []

        line_num = 0
        with open(input_file, "r", encoding="utf-8") as f:
            for line in f:
                line_num += 1
                error_msg = None
                try:
                    record = json.loads(line)
                    if not required_keys.issubset(record.keys()):
                        error_msg = f"Missing required keys. Found: {list(record.keys())}"
                    elif not record.get("label"):
                        error_msg = "Empty label field."
                    elif not record.get("text") or not str(record.get("text")).strip():
                        error_msg = "Empty text field."
                    else:
                        labels_found.add(record.get("label"))

                    if error_msg:
                        error_records.append({
                            "line_num": line_num,
                            "error": error_msg,
                            "raw_line": line.strip()
                        })
                except json.JSONDecodeError:
                    error_records.append({
                        "line_num": line_num,
                        "error": "Invalid JSON.",
                        "raw_line": line.strip()
                    })

        if len(error_records) == 0:
            print(f"Dataset check passed! Validated {line_num} records.")
            print(f"Found {len(labels_found)} unique labels, e.g., {list(labels_found)[:5]}")
        else:
            print(f"Dataset check failed with {len(error_records)} errors.")

            # Save errors to CSV
            output_csv = input_file.replace(".jsonl", "_errors.csv")
            with open(output_csv, "w", encoding="utf-8", newline="") as csvfile:
                fieldnames = ["line_num", "error", "raw_line"]
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()
                for er in error_records:
                    writer.writerow(er)
            print(f"Exported error details to {output_csv}")


Checking datasets in datasets/preprocessed...

--- Checking flores_plus.jsonl ---
Dataset check passed! Validated 223652 records.
Found 206 unique labels, e.g., ['run', 'aeb', 'ekk', 'fra', 'mos']

--- Checking commonlid.jsonl ---
Dataset check passed! Validated 373230 records.
Found 109 unique labels, e.g., ['aeb', 'ukr', 'fra', 'deu', 'por']

--- Checking wili-2018.jsonl ---
Dataset check passed! Validated 235000 records.
Found 235 unique labels, e.g., ['ukr', 'fra', 'nap', 'kbd', 'por']


In [26]:
import joblib
import pandas as pd

MODEL_DIR = "../models"
TARGET_LABELS = {
    "sin": "sinhala",
    "san": "sanskrit",
    "pli": "pali"
}

vectorizer_path = os.path.join(MODEL_DIR, "langid_vectorizer.pkl")
clf_path = os.path.join(MODEL_DIR, "langid_model.pkl")

all_mismatches = []

if not os.path.exists(vectorizer_path) or not os.path.exists(clf_path):
    print("\nBaseline model files not found. Skipping baseline model check.")
else:
    vectorizer = joblib.load(vectorizer_path)
    clf = joblib.load(clf_path)

    dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
    for input_file in dataset_files:
        for dataset_label, model_class in TARGET_LABELS.items():
            records = []
            with open(input_file, "r", encoding="utf-8") as f:
                for line in f:
                    record = json.loads(line)
                    if record.get("label") == dataset_label:
                        records.append(record)

            if not records:
                print(f"\nNo '{dataset_label}'-labeled records in {os.path.basename(input_file)}; skipping baseline check.")
            else:
                df_records = pd.DataFrame(records)
                X = vectorizer.transform(df_records["text"])
                df_records["predicted_label"] = clf.predict(X)

                mismatches = df_records[df_records["predicted_label"] != model_class].reset_index(drop=True)

                print(f"\nBaseline model check on '{dataset_label}' rows in {os.path.basename(input_file)}:")
                print(f"  {len(df_records)} rows labeled '{dataset_label}', {len(mismatches)} not predicted as '{model_class}'")

                if not mismatches.empty:
                    print(mismatches["predicted_label"].value_counts().to_string())

                    checks_dir = "datasets/checks"
                    os.makedirs(checks_dir, exist_ok=True)
                    dataset_name = os.path.splitext(os.path.basename(input_file))[0]
                    mismatches_path = os.path.join(checks_dir, f"{dataset_name}_{model_class}_mismatches.csv")
                    mismatches.to_csv(mismatches_path, index=False)
                    print(f"Saved mismatches to {mismatches_path}")
                    all_mismatches.append(mismatches)

if all_mismatches:
    final_mismatches = pd.concat(all_mismatches, ignore_index=True)
else:
    final_mismatches = pd.DataFrame(columns=["text", "label", "source", "predicted_label"])

final_mismatches


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-li


Baseline model check on 'sin' rows in flores_plus.jsonl:
  1012 rows labeled 'sin', 1 not predicted as 'sinhala'
predicted_label
sanskrit    1
Saved mismatches to datasets/checks/flores_plus_sinhala_mismatches.csv

Baseline model check on 'san' rows in flores_plus.jsonl:
  1012 rows labeled 'san', 1012 not predicted as 'sanskrit'
predicted_label
sinhala    1007
pali          5
Saved mismatches to datasets/checks/flores_plus_sanskrit_mismatches.csv

No 'pli'-labeled records in flores_plus.jsonl; skipping baseline check.

No 'sin'-labeled records in commonlid.jsonl; skipping baseline check.

Baseline model check on 'san' rows in commonlid.jsonl:
  895 rows labeled 'san', 895 not predicted as 'sanskrit'
predicted_label
sinhala    823
pali        72
Saved mismatches to datasets/checks/commonlid_sanskrit_mismatches.csv

No 'pli'-labeled records in commonlid.jsonl; skipping baseline check.

Baseline model check on 'sin' rows in wili-2018.jsonl:
  1000 rows labeled 'sin', 2 not predicted as 

,text,label,source,predicted_label
0,මැස්ලෝගේ අවශ්‍යතා ධූරාවලි න්‍යාය සහ හර්ට්ස්බර්...,sin,flores_plus,sanskrit
1,"""अस्मत्समीपे केचन 4 मासवर्षिणः मधुमेहहीनाः मूष...",san,flores_plus,sinhala
2,"डा. इहद उर् महोदयः, यः हालिफैक्स, नोवा स्कोटिय...",san,flores_plus,sinhala
3,एतेषां निष्कर्षाणां १ प्रकारमधुमेहग्रस्तैः सह ...,san,flores_plus,sinhala
4,सोमवासरे स्वीडिशविहारे साहित्यस्य कृते विद्यमा...,san,flores_plus,sinhala
...,...,...,...,...
2905,कुलशेखरवर्मणा विरचितं नाटकम् अस्ति सुभद्राधनञ्...,san,wili-2018,sinhala
2906,"यदा भगवतः जन्म अभवत्, तदा चतुःषष्टिः इन्द्राः ...",san,wili-2018,sinhala
2907,एतस्मिन् दशपरिच्छेदाः अथवा प्रपाठकाः सन्ति । प...,san,wili-2018,sinhala
2908,कर्णाटकस्य अष्टाविंशतिलोकसभाक्षेत्रेषु अन्यतमम...,san,wili-2018,sinhala


In [27]:
vectorizer_path

'../models/langid_vectorizer.pkl'

In [28]:
import os
import glob
import json
import subprocess
import pandas as pd

# -------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------
# Paste your Google Drive sharing link here:
# Make sure the file is set to "Anyone with the link can view"
GDRIVE_LINK = "https://drive.google.com/drive/folders/1QN6KTRIjGDuu6GTVpZV90f-dU1SNPAnu?usp=sharing" 

# The format of your test dataset: "csv" or "jsonl"
TEST_DATA_FORMAT = "csv" 
# -------------------------------------------------------------

if GDRIVE_LINK != "YOUR_GOOGLE_DRIVE_LINK_HERE":
    print("Downloading test dataset from Google Drive...")
    try:
        import gdown
    except ImportError:
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
        
    if "/folders/" in GDRIVE_LINK:
        print("Detected folder link. Downloading folder...")
        downloaded_files = gdown.download_folder(url=GDRIVE_LINK, output="test_dataset_folder")
        # Find the first valid csv/jsonl in the downloaded folder
        valid_files = [f for f in downloaded_files if f.endswith(f".{TEST_DATA_FORMAT}")]
        if not valid_files:
            raise ValueError(f"No .{TEST_DATA_FORMAT} file found inside the Google Drive folder!")
        test_dataset_path = valid_files[0]
        print(f"Found dataset: {test_dataset_path}")
    else:
        test_dataset_path = "test_dataset." + TEST_DATA_FORMAT
        gdown.download(url=GDRIVE_LINK, output=test_dataset_path)
    
    print(f"\nLoading downloaded test dataset...")
    if TEST_DATA_FORMAT == "csv":
        test_df = pd.read_csv(test_dataset_path)
    else:
        test_df = pd.read_json(test_dataset_path, lines=True)
        
    required_cols = {"text", "label"}
    if not required_cols.issubset(set(test_df.columns)):
        print(f"Error: Test dataset is missing required columns (needs 'text' and 'label'). Found {list(test_df.columns)}")
    else:
        if "source" not in test_df.columns:
            test_df["source"] = "custom_test_dataset"
            
        # Convert long-form labels to pipeline 3-letter codes
        LABEL_MAP = {
            "sinhala": "sin",
            "sanskrit": "san",
            "pali": "pli",
            "sin": "sin",
            "san": "san",
            "pli": "pli"
        }
        test_df["label"] = test_df["label"].str.lower().map(LABEL_MAP).fillna(test_df["label"])
            
        test_records = test_df[["text", "label", "source"]].to_dict("records")
        
        LABELS_TO_REPLACE = {"sin", "san", "pli"}
        test_records_filtered = [r for r in test_records if r.get("label") in LABELS_TO_REPLACE]
        print(f"Found {len(test_records_filtered)} valid test records for Sinhala/Pali/Sanskrit.")
        
        input_dir = 'datasets/preprocessed'
        dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
        
        for input_file in dataset_files:
            original_records = []
            with open(input_file, "r", encoding="utf-8") as f:
                for line in f:
                    original_records.append(json.loads(line))
                    
            filtered_records = [r for r in original_records if r.get("label") not in LABELS_TO_REPLACE]
            new_records = filtered_records + test_records_filtered
            
            with open(input_file, "w", encoding="utf-8") as f:
                for r in new_records:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
                    
            print(f"Replaced {len(original_records) - len(filtered_records)} old records with {len(test_records_filtered)} test records in {os.path.basename(input_file)}")
else:
    print("Please paste your Google Drive link in the GDRIVE_LINK variable to run the replacement.")


Detected folder link. Downloading folder...


Retrieving folder contents


Processing file 1fPmG1r_uhQg4C9QGbPnxPnobp0b39A32 test.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1fPmG1r_uhQg4C9QGbPnxPnobp0b39A32
To: /home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/test_dataset_folder/test.csv
100%|██████████| 9.06M/9.06M [00:02<00:00, 3.67MB/s]
Download completed


Found dataset: test_dataset_folder/test.csv

Loading downloaded test dataset...
Found 7047 valid test records for Sinhala/Pali/Sanskrit.
Replaced 2024 old records with 7047 test records in flores_plus.jsonl
Replaced 895 old records with 7047 test records in commonlid.jsonl
Replaced 2000 old records with 7047 test records in wili-2018.jsonl
